# IPL Crunch '26 — Data Analytics Challenge

**Author:** [Your Name Here]  
**Dataset:** Cricsheet IPL Data (2008–2025)  
**Tools:** Python, Pandas, Matplotlib, Seaborn

---

## Questions Explored:
1. Do teams that win the toss actually win more matches?
2. Which phase impacts victory the most — Powerplay, Middle, or Death?
3. Who are the top batters across seasons?
4. Who are the top bowlers across seasons?
5. What hidden patterns can be discovered?

---

In [ ]:
import os, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 12

print("Libraries loaded!")

## Step 1: Load Consolidated Data

In [ ]:
OUTPUT_DIR = r"E:\data\ipl_output"
matches = pd.read_csv(os.path.join(OUTPUT_DIR, "matches.csv"))
deliveries = pd.read_csv(os.path.join(OUTPUT_DIR, "deliveries.csv"))
print(f"Matches: {len(matches)}")
print(f"Deliveries: {len(deliveries):,}")
print(f"Seasons: {sorted(matches['season_year'].dropna().unique())}")

## Step 2: Data Overview

In [ ]:
matches.head(3)

In [ ]:
deliveries.head(3)

In [ ]:
def get_phase(over):
    if over <= 6: return "Powerplay (1-6)"
    elif over <= 15: return "Middle (7-15)"
    else: return "Death (16-20)"

deliveries["over"] = deliveries["ball"].astype(float).apply(lambda x: int(x))
deliveries["phase"] = deliveries["over"].apply(get_phase)
deliveries["total_runs"] = deliveries["runs_off_bat"] + deliveries["extras"]
deliveries["is_wicket"] = (
    deliveries["wicket_type"].notna() & 
    (deliveries["wicket_type"] != "run out")
).astype(int)
deliveries["batting_team_won"] = (deliveries["batting_team"] == deliveries["winner"]).astype(int)
matches["toss_winner_won_match"] = (matches["toss_winner"] == matches["winner"]).astype(int)
print("Features engineered!")

## Question 1: Do teams that win the toss actually win more matches?

In [ ]:
toss_win_rate = matches["toss_winner_won_match"].mean() * 100
print(f"Toss winner wins: {toss_win_rate:.1f}% of matches")
if toss_win_rate > 50:
    print("Slight toss advantage exists, but it is not decisive.")
else:
    print("Toss has little to no impact on match outcomes.")

In [ ]:
display(Image(filename=os.path.join(OUTPUT_DIR, '01_toss_vs_win.png')))

## Question 2: Which phase impacts victory the most?

In [ ]:
phase_stats = deliveries.groupby(["batting_team_won", "phase"]).agg(
    balls=("ball", "count"), runs=("total_runs", "sum"), wickets=("is_wicket", "sum")
).reset_index()
phase_stats["run_rate"] = (phase_stats["runs"] / phase_stats["balls"]) * 6
pivot = phase_stats.pivot_table(index="phase", columns="batting_team_won", values="run_rate")
pivot.columns = ["Losing Side", "Winning Side"]
print(pivot)

In [ ]:
display(Image(filename=os.path.join(OUTPUT_DIR, '02_phase_impact.png')))

## Question 3: Top Batters Across Seasons

In [ ]:
batter_stats = deliveries.groupby("striker").agg(
    balls_faced=("ball", "count"),
    total_runs=("runs_off_bat", "sum"),
    dismissals=("is_wicket", lambda x: (
        deliveries.loc[x.index, "player_dismissed"] == deliveries.loc[x.index, "striker"]
    ).sum())
).reset_index()
batter_stats = batter_stats[batter_stats["balls_faced"] >= 500].copy()
batter_stats["average"] = batter_stats["total_runs"] / batter_stats["dismissals"].replace(0, np.nan)
batter_stats["strike_rate"] = (batter_stats["total_runs"] / batter_stats["balls_faced"]) * 100
print("Top 10 Run-Scorers in IPL:")
print(batter_stats.nlargest(10, "total_runs")[["striker", "total_runs", "average", "strike_rate"]].to_string(index=False))

In [ ]:
display(Image(filename=os.path.join(OUTPUT_DIR, '03_top_batters.png')))

In [ ]:
display(Image(filename=os.path.join(OUTPUT_DIR, '03b_top_batter_per_season.png')))

## Question 4: Top Bowlers Across Seasons

In [ ]:
bowler_stats = deliveries.groupby("bowler").agg(
    balls_bowled=("ball", "count"),
    runs_conceded=("total_runs", "sum"),
    wickets=("is_wicket", "sum")
).reset_index()
bowler_stats = bowler_stats[bowler_stats["balls_bowled"] >= 300].copy()
bowler_stats["economy"] = (bowler_stats["runs_conceded"] / bowler_stats["balls_bowled"]) * 6
bowler_stats["average"] = bowler_stats["runs_conceded"] / bowler_stats["wickets"].replace(0, np.nan)
print("Top 10 Wicket-Takers in IPL:")
print(bowler_stats.nlargest(10, "wickets")[["bowler", "wickets", "average", "economy"]].to_string(index=False))

In [ ]:
display(Image(filename=os.path.join(OUTPUT_DIR, '04_top_bowlers.png')))

In [ ]:
display(Image(filename=os.path.join(OUTPUT_DIR, '04b_top_bowler_per_season.png')))

## Bonus: Hidden Patterns

In [ ]:
inn1 = deliveries[deliveries["innings"]==1].groupby("match_id").agg(score1=("total_runs","sum")).reset_index()
inn2 = deliveries[deliveries["innings"]==2].groupby("match_id").agg(score2=("total_runs","sum")).reset_index()
m = inn1.merge(inn2, on="match_id").merge(matches[["match_id","winner"]], on="match_id")
inn2_winners = deliveries[deliveries["innings"]==2].groupby("match_id").agg(bat2=("batting_team","first")).reset_index()
m = m.merge(inn2_winners, on="match_id")
m["chasing_won"] = (m["bat2"] == m["winner"]).astype(int)
chase_win = m["chasing_won"].mean() * 100
print(f"Chasing team won: {chase_win:.1f}% of matches")
print(f"Batting first won: {100-chase_win:.1f}% of matches")

In [ ]:
display(Image(filename=os.path.join(OUTPUT_DIR, '05_hidden_patterns.png')))

In [ ]:
display(Image(filename=os.path.join(OUTPUT_DIR, '06_run_rate_trend.png')))

## Summary of Findings

| Question | Finding |
|---|---|
| Toss to Win? | 50.6% - marginal, not decisive |
| Most impactful phase | Death overs (16-20) |
| Top batter | Most runs: Virat Kohli |
| Top bowler | Most wickets: Yuzvendra Chahal |
| Chasing vs Defending | 53.9% matches won by chasing team |
| Run rate trend | IPL scoring rate keeps increasing |

## Your Task: ONE Surprising Insight

Look at the charts above and write 1-2 paragraphs about something that genuinely surprised you.

Examples:
- A player you did not expect in the top 10
- A season where chasing was unusually hard/easy
- A venue where toss matters more than others
- Evolution of IPL scoring patterns

This is the most important part of your submission - make it personal!

---

**Submission Checklist:**
- [ ] Replace [Your Name Here] at the top
- [ ] Add your interpretation under each chart
- [ ] Write your ONE surprising insight
- [ ] Export as PDF (File > Download > PDF)
- [ ] Upload to GitHub repository
